# Fetal Head Segmentation – Result Visualisation

In [ ]:
import shutil

import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import PIL
import rootutils
import torch
import torch.nn.functional as F
import torchvision.transforms.v2 as T
import torchvision.transforms.v2.functional as TF
from torchvision.io import read_image
from torchvision.ops import masks_to_boxes
from tqdm.notebook import tqdm

root = rootutils.setup_root(search_from=".", indicator=".project-root", pythonpath=True)

from src.data.components.dataset import (
    USVideosFrameDataset,
    USVideosSsimFrameDataset,
)
from src.data.components.transforms import PadToAspectRatio, Resize
from src.data.utils.utils import (
    create_ellipse_tensor,
    crop,
    find_angle,
    get_dice_score,
    read_image_tensor,
    show_pytorch_images,
)
from src.models.head_segmentation_module import HeadSegmentationLitModule

data_dir = root / "data"
dataset_name = "FETAL_HEAD_SEGMENTATION"
dataset_dir = data_dir / dataset_name

## Enable Interactive Matplotlib Backend

Switch to the `ipympl` backend so that figures are rendered as interactive widgets with pan, zoom, and hover support.
Run this cell once at the start of the session.

In [ ]:
%matplotlib ipympl

import matplotlib

matplotlib.use("ipympl")

## Reset Matplotlib Backend

Run this cell if figures stop rendering or become unresponsive.
Closes all open figures and restarts the `ipympl` interactive backend.

In [ ]:
# First, close any open figures
plt.close("all")

# Then, restart the ipympl backend by rerunning the magic command
%matplotlib ipympl

# Prediction Error Analysis

Load per-image prediction results from a saved `prediction.csv` alongside the ground-truth labels from `data.csv` and `data_fix.csv`.
No live model is required — this section works from pre-computed predictions.

In [ ]:
data_df = pd.read_csv(dataset_dir / "data.csv", dtype={"Patient_num": str})
fetal_planes = pd.read_csv(f"{data_dir}/FETAL_PLANES/data_fix.csv")
predictions_df = pd.read_csv(dataset_dir / "prediction.csv")
len(predictions_df)

## Unidentified Brain Images

Brain-plane images the model failed to detect — i.e. rows where the ground truth is brain but `Prediction == 0`.
Up to 25 samples are displayed for visual inspection.

In [ ]:
misidentified_images = predictions_df[predictions_df["Prediction"] == 0]
misidentified_images = misidentified_images.sort_values(["Count", "Image_name"], ascending=[False, True])
print(len(misidentified_images))

image_size = (192, 256)
pad_to_aspect_ration = PadToAspectRatio(image_size, fill=0)

images = []
ignore = [
    # Ignore Not a Brain images
    # numbers
    # Ignore Brain images
    # ommited for now
]

for _, misidentified_row in tqdm(misidentified_images.iterrows(), total=len(misidentified_images)):
    for index, row in fetal_planes[fetal_planes["Image_name"] == misidentified_row["Image_name"]].iterrows():
        for _, data_row in data_df[data_df["Image_name"] == misidentified_row["Image_name"]].iterrows():
            if index in ignore or data_row["Brain_plane"] == 0:
                continue
            image_name = row["Image_name"]
            plane = row["Plane"]
            brain_plane = row["Brain_plane"]
            subset = row["Subset"]

            image_path = f"{data_dir}/FETAL_PLANES/Images/{image_name}.png"
            image = read_image_tensor(image_path)
            image = pad_to_aspect_ration(image)
            images.append((image, f"{index}: {brain_plane} ({misidentified_row["Count"]})"))
            # print(f"{index:<6}: {image_name:<30} {brain_plane:<20} {misidentified_row["Count"]}")

print(len(images))
if images:
    show_pytorch_images(images[:25], gray=False).show()

## False Positive Brain Detections

Non-brain images the model incorrectly classified as containing a head (`Prediction == 1`).
Up to 16 samples are displayed; image names are printed for review.

In [ ]:
misidentified_images = predictions_df[predictions_df["Prediction"] == 1]
misidentified_images = misidentified_images.sort_values(["Count", "Image_name"], ascending=[False, True])
print(len(misidentified_images))

image_size = (192, 256)
pad_to_aspect_ration = PadToAspectRatio(image_size, fill=0)

images = []
ignore = [
    # Ignore Not a Brain images
    # Ignore Brain images
    # ommited for now
]

limit = 16

for _, misidentified_row in tqdm(misidentified_images.iterrows(), total=len(misidentified_images)):
    for index, row in fetal_planes[fetal_planes["Image_name"] == misidentified_row["Image_name"]].iterrows():
        for _, data_row in data_df[data_df["Image_name"] == misidentified_row["Image_name"]].iterrows():
            if index in ignore or data_row["Brain_plane"] == 1:
                continue
            image_name = row["Image_name"]
            plane = row["Plane"]
            brain_plane = row["Brain_plane"]
            subset = row["Subset"]

            image_path = f"{data_dir}/FETAL_PLANES/Images/{image_name}.png"
            image = read_image_tensor(image_path)
            image = pad_to_aspect_ration(image)
            # images.append((image, f"{index}: {plane} ({misidentified_row["Count"]})"))
            images.append((image, f"{index}: ({misidentified_row["Count"]})"))
            # print(f"{index}, ")
            print(f"{index:<6}: {image_name:<30} {plane:<18} {misidentified_row["Count"]}")

            if len(images) >= limit:
                break
        if len(images) >= limit:
            break
    if len(images) >= limit:
        break

print(len(images))
if images:
    show_pytorch_images(images[:limit], gray=False).show()

# Model Setup

Load the trained `HeadSegmentationLitModule` from the most recent checkpoint.

In [ ]:
# checkpoint_file = root / "logs" / "head_segmentation_train" / "runs" / "2025-08-25_13-36-03" # cerulean-wave-694 42/8 - best
checkpoint_file = root / "logs" / "head_segmentation_train" / "runs" / "2025-10-13_15-27-50"
checkpoint = sorted(checkpoint_file.glob("checkpoints/epoch_*.ckpt"))[-1]
model = HeadSegmentationLitModule.load_from_checkpoint(str(checkpoint), weights_only=False)
# disable randomness, dropout, etc...
model.eval()
model.to("cpu")
print("Loaded")

## Inference Helpers

Define the functions and transforms used for live model inference:
- `predict_head_mask` — runs the model on a single image tensor and returns a binary mask plus a detection label
- `overlap_mask` — blends the predicted mask over the original image for visualisation
- `PadToAspectRatio` — pads an image to a fixed aspect ratio before passing it to the model
- `transform` — the inference preprocessing pipeline (resize → pad → normalise)

In [ ]:
def predict_head_mask(model: HeadSegmentationLitModule, image: torch.Tensor) -> tuple[torch.Tensor, int]:
    """Binary mask [1, H, W] and frame label (same rules as HeadSegmentationLitModule.calculate_prediction)."""
    logits = model(image.unsqueeze(0))
    binary_mask, prediction_labels = model.calculate_hard_prediction(logits)
    return binary_mask.squeeze(0), int(prediction_labels.squeeze(0).item())


def overlap_mask(image: torch.Tensor, mask: torch.Tensor):
    image = image.clone()
    if image.shape[0] == 1:
        image = image.repeat(3, 1, 1)
    if image.max() <= 1.0:
        image = image * 255

    image = image.int()
    image[0] = image[0] + mask * 255 * 0.4
    image = torch.clamp(image, min=0, max=255)
    image = image.to(torch.uint8)

    return image


image_size = (192, 256)
transform = torch.nn.Sequential(
    T.Grayscale(),
    PadToAspectRatio(image_size, fill=0),
    Resize(image_size, interpolation=T.InterpolationMode.NEAREST),
    T.ToDtype(torch.float32, scale=True),
)
pad_to_aspect_ration = PadToAspectRatio(image_size, fill=0)

# FETAL_PLANES Dataset Evaluation

Run the loaded model over every image in FETAL_PLANES and categorise the results into five buckets for further inspection.

## Evaluation Pipeline

For each image the model produces a binary head mask and a frame-level detection label.
Results are sorted into five lists:
- `brain_images` — correctly detected brain frames with high dice score (> 0.95 vs fitted ellipse)
- `brain_low_score_images` — correctly detected but with lower dice score (≤ 0.95)
- `unidentified_images` — brain frames the model missed (`prediction_label == 0`)
- `misidentified_images` — non-brain frames predicted as head (`prediction_label == 1`)

Counts for each bucket are printed after the loop.

In [ ]:
head_segmentation = pd.read_csv(f"{dataset_dir}/data.csv")
fetal_planes = pd.read_csv(f"{data_dir}/FETAL_PLANES/data.csv")

brain_images = []
brain_high_score_images = []
brain_low_score_images = []
unidentified_images = []
misidentified_images = []
for index, plane_row in tqdm(fetal_planes.iterrows(), total=len(fetal_planes), desc="Process images"):
    for _, row in head_segmentation[head_segmentation["Image_name"] == plane_row["Image_name"]].iterrows():
        if row["Valid"] != 1:
            continue

        is_brain_plane = row["Brain_plane"] == 1
        image_path = f"{dataset_dir}/{row["Ultrasound_path"]}"
        image = read_image_tensor(image_path)

        with torch.no_grad():
            brain_plane = transform(image)
            prediction, prediction_label = predict_head_mask(model, brain_plane)

            if prediction_label == 1:
                brain_plane = pad_to_aspect_ration(image)
                angle = find_angle(prediction)
                prediction = TF.resize(
                    prediction, size=brain_plane.shape[1:], interpolation=T.InterpolationMode.NEAREST
                )
                brain_plane = overlap_mask(brain_plane, prediction)

                prediction = TF.rotate(prediction, angle=angle, expand=True, interpolation=T.InterpolationMode.NEAREST)

                x1, y1, x2, y2 = masks_to_boxes(prediction)[0].int()
                perfect_mask = create_ellipse_tensor(
                    prediction.shape[1],
                    prediction.shape[2],
                    x1 + ((x2 - x1) / 2),
                    y1 + ((y2 - y1) / 2),
                    (x2 - x1) / 2,
                    (y2 - y1) / 2,
                    add_half=True,
                )
                dice_score = get_dice_score(prediction, perfect_mask)

                if is_brain_plane:
                    brain_images.append((index, brain_plane))

                    if dice_score > 0.95:
                        brain_high_score_images.append((index, brain_plane))
                    else:
                        brain_low_score_images.append((index, brain_plane))

                else:
                    misidentified_images.append((index, brain_plane))
            else:
                if is_brain_plane:
                    unidentified_images.append((index, image))


print(f"Number of brain images {len(brain_images)}")
print(f"Number of high score brain images {len(brain_high_score_images)}")
print(f"Number of low score brain images {len(brain_low_score_images)}")
print(f"Number of unidentified images {len(unidentified_images)}")
print(f"Number of misidentified images {len(misidentified_images)}")

### Random Brain Images

Display a random sample of up to 64 correctly detected brain images with the predicted mask blended over the original.

In [ ]:
images = []
for rand_i in torch.randperm(len(brain_images))[:64]:
    i, image = brain_images[rand_i]
    images.append(image)
    # image_name = fetal_planes.Image_name[i]
    # plane = fetal_planes.Plane[i]
    # brain_plane = fetal_planes.Brain_plane[i]
    # subset = fetal_planes.Subset[i]
    # images.append((image, f"Brain_{brain_plane} - {subset}"))

if images:
    show_pytorch_images(images, gray=False).show()

### Low-Score Brain Images

Brain images that were detected correctly but whose predicted mask has a dice score ≤ 0.95 against a fitted ellipse.
Up to 64 samples are shown; these are candidates for retraining or mask correction.

In [ ]:
images = []
for i, image in brain_low_score_images[:64]:
    # image_name = fetal_planes.Image_name[i]
    # image_path = images_dir / f"{image_name}.png"
    # image_original = read_image_tensor(image_path)
    # image_original = pad_to_aspect_ration(image_original)
    # plane = fetal_planes.Plane[i]
    # brain_plane = fetal_planes.Brain_plane[i]
    # subset = fetal_planes.Subset[i]
    # images.append((image_original, f"{i}:{brain_plane}"))
    images.append(image)

if images:
    show_pytorch_images(images, gray=False).show()

### Unidentified Brain Images

Brain-plane images the live model failed to detect (`prediction_label == 0`).
Up to 64 samples are padded and displayed for manual review.

In [ ]:
images = []
for i, image in unidentified_images[:64]:
    image = pad_to_aspect_ration(image)
    # image_name = fetal_planes.Image_name[i]
    # plane = fetal_planes.Plane[i]
    # brain_plane = fetal_planes.Brain_plane[i]
    # subset = fetal_planes.Subset[i]
    # images.append((image, f"{i}:Brain_{brain_plane} - {subset}"))
    # print(f"{i}: {image_name}")
    images.append(image)

if images:
    show_pytorch_images(images).show()

### False Positive Detections

Non-brain images classified as containing a head by the live model.
Image names and plane labels are printed; the list can be used to extend the exclusion list in `data_fix.csv`.

In [ ]:
images = []
for i, image in misidentified_images:
    image_name = fetal_planes.Image_name[i]
    plane = fetal_planes.Plane[i]
    brain_plane = fetal_planes.Brain_plane[i]
    subset = fetal_planes.Subset[i]
    images.append((image, f"{i}:{plane}"))
    print(f'"{image_name}",')

if images:
    show_pytorch_images(images, gray=False).show()

### Dice Score Analysis for False Positives

Re-run inference on each false positive and compute a dice score between the predicted mask and a fitted ellipse.
Results are split into two groups — score > 0.95 and score ≤ 0.95 — to distinguish confident misclassifications from noisy predictions.

In [ ]:
score_00_images = []
score_95_images = []
for i, _ in misidentified_images:
    plane = fetal_planes.Plane[i]
    image_name = fetal_planes.Image_name[i]
    image_path = data_dir / "FETAL_PLANES" / "Images" / f"{image_name}.png"
    image = read_image_tensor(image_path)

    with torch.no_grad():
        brain_plane_tensor = transform(image)
        prediction, prediction_label = predict_head_mask(model, brain_plane_tensor)

        if prediction_label == 1:
            brain_plane = pad_to_aspect_ration(image)
            angle = find_angle(prediction)
            prediction = TF.resize(prediction, size=brain_plane.shape[1:], interpolation=T.InterpolationMode.NEAREST)
            brain_plane = overlap_mask(brain_plane, prediction)

            prediction = TF.rotate(prediction, angle=angle, interpolation=T.InterpolationMode.NEAREST)

            x1, y1, x2, y2 = masks_to_boxes(prediction)[0].int()
            perfect_mask = create_ellipse_tensor(
                prediction.shape[1],
                prediction.shape[2],
                x1 + ((x2 - x1) / 2),
                y1 + ((y2 - y1) / 2),
                (x2 - x1) / 2,
                (y2 - y1) / 2,
                add_half=True,
            )
            dice_score = get_dice_score(prediction, perfect_mask)

            if dice_score > 0.95:
                score_95_images.append((brain_plane, f"{dice_score.item():.3f}"))
            else:
                score_00_images.append((image, f"{i}:{plane}"))
                score_00_images.append((brain_plane, f"{dice_score.item():.3f}"))

print(f"Number of 0.00 {len(score_00_images) // 2}/{len(misidentified_images)}")
print(f"Number of 0.95 {len(score_95_images)}/{len(misidentified_images)}")

if score_00_images:
    show_pytorch_images(score_00_images, gray=False).show()
if score_95_images:
    show_pytorch_images(score_95_images, gray=False).show()

# Mismatch Verification with Model Ensemble

Verify whether a list of suspected-mislabelled images are *genuinely* misclassified, using an
ensemble of models that were **trained without the mismatch data**.

Provide two inputs below:
- `mismatch_image_names` — a list of `Image_name`s (from FETAL_PLANES) to inspect.
- `checkpoint_paths` — a list of full `.ckpt` paths. Using several checkpoints makes the
  verdict more robust than a single model.

For every image each model predicts a binary head mask and a brain/not-brain label. The models are
combined by **majority vote on the label** (ties and the `0.5` boundary resolve to brain = 1) and a
**mean dice** over the models that detected a head.

> **Note on the dice score:** these images have hand-annotated ground-truth masks stored at
> `FETAL_HEAD_SEGMENTATION/Fix/Segmentations/{Image_name}.png`. The reported dice is the true Dice
> coefficient between each predicted mask and that ground-truth mask (both resized to the model
> input resolution). Images without a mask file report `NaN` dice.

**Prerequisite:** run the *Inference Helpers* cell first (it defines `transform`,
`pad_to_aspect_ration`, `overlap_mask` and `predict_head_mask`). This section loads its own models,
so the single-model *Model Setup* cell is not required.

## Inputs and Model Ensemble

Fill in the suspected-mismatch image names and the checkpoint paths to validate against, then load
every checkpoint into an evaluation-mode model on CPU.

In [ ]:
# Image_name values (without extension) of the suspected-mislabelled images to verify.
mismatch_image_names = [
    # "Patient00123_Plane3_4_of_7",
]

# Full paths to the checkpoints to validate against (models trained WITHOUT the mismatch data).
checkpoint_paths = [
    # root / "logs" / "head_segmentation_train" / "runs" / "2025-10-13_15-27-50" / "checkpoints" / "epoch_000.ckpt",
]

models = []
for checkpoint_path in checkpoint_paths:
    checkpoint = max(checkpoint_path.glob("checkpoints/epoch_*.ckpt"))
    ensemble_model = HeadSegmentationLitModule.load_from_checkpoint(str(checkpoint), weights_only=False)
    ensemble_model.eval()
    ensemble_model.to("cpu")
    models.append(ensemble_model)

print(f"Loaded {len(models)} model(s) for {len(mismatch_image_names)} mismatch image(s)")

## Ensemble Inference and Results

For each image, run every model, aggregate the labels by majority vote and average the dice against
the hand-annotated ground-truth mask across all models. The results are collected into a DataFrame
and the predicted masks are overlaid for visual inspection.

Each caption reads `vote_label (agreement) d=mean_dice`, where `vote_label` is 1 (brain) or 0 (not a
brain), `agreement` is the share of models that agreed with the vote, and `mean_dice` is the average
Dice between the predicted masks and the ground-truth mask (`NaN` when no mask file exists).

In [ ]:
def label_to_int(brain_plane) -> int:
    """Map a FETAL_PLANES Brain_plane value to 1 (brain) or 0 (not a brain)."""
    return 0 if str(brain_plane) == "Not A Brain" else 1


# Geometry-only transform that maps a mask into the model input space (matches `transform`).
mask_transform = torch.nn.Sequential(
    PadToAspectRatio(image_size, fill=0),
    Resize(image_size, interpolation=T.InterpolationMode.NEAREST),
)
true_mask_dir = dataset_dir / "Fix" / "Segmentations"


def load_true_mask(image_name: str) -> torch.Tensor | None:
    """Hand-annotated mask resized to the model input space, as a [1, H, W] {0, 1} tensor."""
    mask_path = true_mask_dir / f"{image_name}.png"
    if not mask_path.exists():
        return None
    mask = read_image(str(mask_path))[:1, :, :] // 255  # single-channel binary
    mask = mask_transform(mask)
    return mask.float()


results = []
result_images = []
for image_name in tqdm(mismatch_image_names, desc="Verify mismatch images"):
    image_path = data_dir / "FETAL_PLANES" / "Images" / f"{image_name}.png"
    image = read_image_tensor(image_path)

    plane_rows = fetal_planes[fetal_planes["Image_name"] == image_name]
    gt_label = label_to_int(plane_rows.iloc[0]["Brain_plane"]) if len(plane_rows) else None

    true_mask = load_true_mask(image_name)

    model_labels = []
    model_dices = []
    overlay_image = None
    for model in models:
        with torch.no_grad():
            brain_plane_tensor = transform(image)
            prediction, prediction_label = predict_head_mask(model, brain_plane_tensor)

        model_labels.append(prediction_label)
        if true_mask is not None:
            model_dices.append(get_dice_score(prediction, true_mask).item())
        if prediction_label == 1 and overlay_image is None:
            padded = pad_to_aspect_ration(image)
            mask = TF.resize(prediction, size=padded.shape[1:], interpolation=T.InterpolationMode.NEAREST)
            overlay_image = overlap_mask(padded, mask)

    num_models = len(model_labels)
    brain_votes = sum(model_labels)
    # Majority vote; ties and the 0.5 boundary resolve to brain (1).
    vote_label = 1 if (num_models and brain_votes / num_models >= 0.5) else 0
    agreement = (brain_votes if vote_label == 1 else num_models - brain_votes) / num_models if num_models else 0.0
    mean_dice = sum(model_dices) / len(model_dices) if model_dices else float("nan")

    results.append(
        {
            "Image_name": image_name,
            "GT_label": gt_label,
            "Vote_label": vote_label,
            "Agreement": agreement,
            "Mean_dice": mean_dice,
            "Has_true_mask": true_mask is not None,
            "Num_detected": brain_votes,
            "Num_models": num_models,
            "Labels": tuple(model_labels),
        }
    )

    if overlay_image is None:
        overlay_image = pad_to_aspect_ration(image)
    result_images.append((overlay_image, f"{vote_label} ({agreement:.0%}) d={mean_dice:.3f}"))

results_df = pd.DataFrame(results)
if len(results_df):
    results_df = results_df.sort_values(["Vote_label", "Agreement", "Mean_dice"], ascending=[True, False, False])
    voted_brain = int((results_df["Vote_label"] == 1).sum())
    print(f"Voted brain: {voted_brain}/{len(results_df)} | voted not-a-brain: {len(results_df) - voted_brain}")
    missing_mask = int((~results_df["Has_true_mask"]).sum())
    if missing_mask:
        print(f"Missing true mask for {missing_mask}/{len(results_df)} image(s) (dice is NaN for those)")
    if results_df["GT_label"].notna().any():
        disagree = int((results_df["Vote_label"] != results_df["GT_label"]).sum())
        print(f"Disagree with ground truth: {disagree}/{len(results_df)}")

if result_images:
    show_pytorch_images(result_images, gray=False).show()

results_df

# Video Processing with Model Inference

Apply the head-segmentation model frame-by-frame to SSIM-subsampled ultrasound videos.
For each detected head, the frame is rotated to align the principal axis and cropped to the bounding box.
An EMA stabiliser smooths the bounding-box coordinates and rotation angle across frames to reduce jitter.

## Raw Video Playback

Load the SSIM-subsampled video dataset and animate a single video using `matplotlib.animation.ArtistAnimation` as a quick sanity-check before running inference.

In [ ]:
videos = USVideosSsimFrameDataset(
    data_dir=data_dir,
    dataset_name="US_VIDEOS_ssim_0.7",
    transform=torch.nn.Sequential(
        T.Grayscale(),
        T.Resize((165, 240), interpolation=T.InterpolationMode.NEAREST),
        T.ToDtype(dtype=torch.float32, scale=True),
    ),
)

fig, ax = plt.subplots()

video = videos[1]
frame = video[0]
frame = frame.detach()
frame = TF.to_pil_image(frame)
frame = TF.to_grayscale(frame)
ax.imshow(np.asarray(frame), cmap="gray")
ax.axis("off")

frames = []
for idx in tqdm(range(len(video))):
    frame = video[idx]
    frame = frame.detach()
    frame = TF.to_pil_image(frame)
    frame = TF.to_grayscale(frame)
    frame = ax.imshow(np.asarray(frame), cmap="gray", animated=True)
    frames.append([frame])

ani = animation.ArtistAnimation(fig, frames, interval=50, blit=True, repeat_delay=1000)
plt.show()

## Animation Helpers

Define `display_image_animation` and `display_overlap_mask_animation` for rendering per-frame overlays in a matplotlib animation.
Also sets up the dataset objects used in the animation and export cells below.

In [ ]:
def display_image_animation(image, ax, title):
    image = image.detach()
    image = TF.to_pil_image(image)
    image = TF.to_grayscale(image)
    image = np.asarray(image)

    if not ax.images:
        ax.imshow(image, cmap="gray")
        ax.set_title(title)
        ax.axis("off")

    return ax.imshow(image, cmap="gray", animated=True)


def display_overlap_mask_animation(image: torch.Tensor, mask: torch.Tensor, ax, title):
    image = image.clone()
    image = TF.to_grayscale(image)
    if image.shape[0] == 1:
        image = image.repeat(3, 1, 1)
    if image.max() <= 1.0:
        image = image * 255

    image = image.int()
    image[0] = image[0] + mask * 255 * 0.4
    image = torch.clamp(image, min=0, max=255)
    image = image.to(torch.uint8)

    image = TF.to_image(image)
    image = image.permute(1, 2, 0).numpy()

    if not ax.images:
        ax.imshow(image)
        ax.set_title(title)
        ax.axis("off")

    return ax.imshow(image, animated=True)


videos = USVideosSsimFrameDataset(
    data_dir=data_dir,
    dataset_name="US_VIDEOS_ssim_0.7",
    transform=torch.nn.Sequential(
        T.Grayscale(),
        T.Resize((165, 240), interpolation=T.InterpolationMode.NEAREST),
        T.ToDtype(dtype=torch.float32, scale=True),
    ),
)

# fig, axes = plt.subplots(ncols=3, nrows=1, squeeze=False, figsize=(10, 4))

# video = videos[1]
# frames = []
# for idx in tqdm(range(len(video))):
#     frame = video[idx]
#     with torch.no_grad():
#         logits = model(frame.unsqueeze(0)).squeeze(0)
#         prediction = (torch.sigmoid(logits) > 0.5).float()

#     im1 = display_image_animation(frame, axes[0, 0], "Image")
#     im2 = display_image_animation(prediction, axes[0, 1], "Prediction")
#     im3 = display_overlap_mask_animation(frame, prediction, axes[0, 2], "Overlayed Mask")
#     frames.append([im1, im2, im3])

# ani = animation.ArtistAnimation(fig, frames, interval=100, blit=True, repeat_delay=1000)
# plt.tight_layout()
# plt.show()

## Stabilised Crop Animation

Define `StabilizeVideoEma`, which applies exponential moving average smoothing to the detected bounding-box coordinates and rotation angle across frames.
The class is used in the export pipeline to reduce per-frame jitter.

In [ ]:
class StabilizeVideoEma:
    def __init__(self, alpha: float = 0.5):
        self.alpha = alpha
        self.args = None
        self.int_type = False

    def stabilize(self, *args):
        if self.args is None:
            self.args = self._extract_item_from_tensor(args)

            for arg in self.args:
                if isinstance(arg, int):
                    self.int_type = True
        else:
            for i in range(len(self.args)):
                self.args[i] = self.alpha * args[i] + (1 - self.alpha) * self.args[i]

        rs = self._extract_item_from_tensor(self.args)
        rs = [int(round(arg)) if self.int_type else arg for arg in rs]

        if len(rs) == 1:
            return rs[0]
        else:
            return rs

    def _extract_item_from_tensor(self, args):
        return [args[i].item() if isinstance(args[i], torch.Tensor) else args[i] for i in range(len(args))]

    def reset(self):
        self.args = None
        self.int_type = False

### Padding Helper

`pad_central` pads a tensor to a target height × width by adding equal margins on each side, keeping the original content centred.
Used to ensure every video frame is the same size before the crop-export loop.

In [ ]:
def pad_central(image, height, width):
    original_height, original_width = image.shape[1:]

    padding_needed = height - original_height
    top_pad = padding_needed // 2
    bottom_pad = padding_needed - top_pad

    padding_needed = width - original_width
    left_pad = padding_needed // 2
    right_pad = padding_needed - left_pad

    return TF.pad(image, padding=[left_pad, top_pad, right_pad, bottom_pad], fill=0)


videos = USVideosFrameDataset(
    data_dir=data_dir,
    train=False,
    transform=torch.nn.Sequential(
        T.Grayscale(),
        T.ToDtype(dtype=torch.float32, scale=True),
    ),
)

fig, axes = plt.subplots(ncols=3, nrows=3, squeeze=False, figsize=(10, 8))
video = videos[0]
frames = []

image_size = (165, 240)
interpolation = T.InterpolationMode.NEAREST

frame_crop_prev = None
frame_rotate_crop_prev = None

alpha = 0.5
ema_stabilizer_crop = StabilizeVideoEma(alpha=alpha)
ema_stabilizer_rotate = StabilizeVideoEma(alpha=alpha)
ema_stabilizer_rotate_crop = StabilizeVideoEma(alpha=alpha)

alpha = 0.1
ema_stabilizer_crop_2 = StabilizeVideoEma(alpha=alpha)
ema_stabilizer_rotate_2 = StabilizeVideoEma(alpha=alpha)
ema_stabilizer_rotate_crop_2 = StabilizeVideoEma(alpha=alpha)

pad = 5
image_pad_central = (700, 900)
# image_size_crop=(175, 225)
image_size_crop = (88, 112)

for idx in tqdm(range(len(video))):
    frame_base = video[idx]
    frame = TF.resize(frame_base, size=image_size, interpolation=interpolation)
    with torch.no_grad():
        prediction, _ = predict_head_mask(model, frame)
        prediction_base = TF.resize(prediction, size=frame_base.shape[1:], interpolation=interpolation)

    # --------------------------------------------
    im1 = display_image_animation(frame, axes[0, 0], "Image")
    im2 = display_image_animation(prediction, axes[0, 1], "Prediction")
    im3 = display_overlap_mask_animation(frame, prediction, axes[0, 2], "Overlayed Mask")

    # --------------------------------------------
    x1, y1, x2, y2 = masks_to_boxes(prediction_base)[0].int()
    frame_2 = crop(frame_base, x1, y1, x2, y2, pad=pad)
    frame_2 = pad_central(frame_2, *image_pad_central)
    frame_2 = TF.resize(frame_2, size=image_size_crop, interpolation=interpolation)
    im4 = display_image_animation(frame_2, axes[1, 0], "Crop")

    # --------------------------------------------
    x1, y1, x2, y2 = masks_to_boxes(prediction_base)[0].int()
    x1, y1, x2, y2 = ema_stabilizer_crop.stabilize(x1, y1, x2, y2)
    frame_2 = crop(frame_base, x1, y1, x2, y2, pad=pad)
    frame_2 = pad_central(frame_2, *image_pad_central)
    frame_2 = TF.resize(frame_2, size=image_size_crop, interpolation=interpolation)
    im5 = display_image_animation(frame_2, axes[1, 1], "ema-stabilized")

    # --------------------------------------------
    x1, y1, x2, y2 = masks_to_boxes(prediction_base)[0].int()
    x1, y1, x2, y2 = ema_stabilizer_crop_2.stabilize(x1, y1, x2, y2)
    frame_2 = crop(frame_base, x1, y1, x2, y2, pad=pad)
    frame_2 = pad_central(frame_2, *image_pad_central)
    frame_2 = TF.resize(frame_2, size=image_size_crop, interpolation=interpolation)
    im6 = display_image_animation(frame_2, axes[1, 2], "ema-stabilized")

    # --------------------------------------------
    angle = find_angle(prediction_base)
    frame_2 = TF.rotate(frame_base, angle=angle, interpolation=T.InterpolationMode.BILINEAR)
    prediction_2 = TF.rotate(prediction_base, angle=angle, interpolation=T.InterpolationMode.NEAREST)

    x1, y1, x2, y2 = masks_to_boxes(prediction_2)[0].int()
    frame_2 = crop(frame_2, x1, y1, x2, y2, pad=pad)
    frame_2 = pad_central(frame_2, *image_pad_central)
    frame_2 = TF.resize(frame_2, size=image_size_crop, interpolation=interpolation)
    im7 = display_image_animation(frame_2, axes[2, 0], "Rotate-crop")

    # --------------------------------------------
    angle = find_angle(prediction_base)
    angle = ema_stabilizer_rotate.stabilize(angle)
    frame_2 = TF.rotate(frame_base, angle=angle, interpolation=T.InterpolationMode.BILINEAR)
    prediction_2 = TF.rotate(prediction_base, angle=angle, interpolation=T.InterpolationMode.NEAREST)

    x1, y1, x2, y2 = masks_to_boxes(prediction_2)[0].int()
    x1, y1, x2, y2 = ema_stabilizer_rotate_crop.stabilize(x1, y1, x2, y2)
    frame_2 = crop(frame_2, x1, y1, x2, y2, pad=pad)
    frame_2 = pad_central(frame_2, *image_pad_central)
    frame_2 = TF.resize(frame_2, size=image_size_crop, interpolation=interpolation)
    im8 = display_image_animation(frame_2, axes[2, 1], "ema-stabilized")

    # --------------------------------------------
    angle = find_angle(prediction_base)
    angle = ema_stabilizer_rotate_2.stabilize(angle)
    frame_2 = TF.rotate(frame_base, angle=angle, interpolation=T.InterpolationMode.BILINEAR)
    prediction_2 = TF.rotate(prediction_base, angle=angle, interpolation=T.InterpolationMode.NEAREST)

    x1, y1, x2, y2 = masks_to_boxes(prediction_2)[0].int()
    x1, y1, x2, y2 = ema_stabilizer_rotate_crop_2.stabilize(x1, y1, x2, y2)
    frame_2 = crop(frame_2, x1, y1, x2, y2, pad=pad)
    frame_2 = pad_central(frame_2, *image_pad_central)
    frame_2 = TF.resize(frame_2, size=image_size_crop, interpolation=interpolation)
    im9 = display_image_animation(frame_2, axes[2, 2], "ema-stabilized")

    # --------------------------------------------
    frames.append([im1, im2, im3, im4, im5, im6, im7, im8, im9])

ani = animation.ArtistAnimation(fig, frames, interval=10, blit=True, repeat_delay=1000)
plt.tight_layout()
plt.show()

# Video Frame Crop Export

Batch pipeline that processes every frame in `US_VIDEOS_ssim_0.7` and writes cropped head clips to `US_VIDEOS_ssim_0.7_crop`:

1. Run the segmentation model on each frame.
2. If a head is detected: rotate the frame to align the principal axis, apply EMA stabilisation, crop to the bounding box, and save to the output directory.
3. If no head is detected: skip the frame and reset the EMA state.

The output directory is deleted and recreated at the start to ensure a clean export.

In [ ]:
ssim_dataset_path = data_dir / "US_VIDEOS_ssim_0.7"
ssim_crop_dataset_path = data_dir / "US_VIDEOS_ssim_0.7_crop"

ssim_videos_path = ssim_dataset_path / "images"
ssim_crop_videos_path = ssim_crop_dataset_path / "images"

shutil.rmtree(ssim_crop_dataset_path)
ssim_crop_dataset_path.mkdir(parents=True)
ssim_crop_videos_path.mkdir()


alpha = 0.1
stabilizer_rotate = StabilizeVideoEma(alpha=alpha)
stabilizer_crop = StabilizeVideoEma(alpha=alpha)

saved_images = 0
for video_dir in tqdm(list(ssim_videos_path.iterdir()), desc="Videos"):

    output_dir = ssim_crop_videos_path / video_dir.name
    output_dir.mkdir(parents=True)

    for image_path in tqdm(list(video_dir.iterdir()), desc="Video images", leave=False):
        image = read_image(image_path)
        if image.shape[0] == 1:
            image = image.repeat(3, 1, 1)
        elif image.shape[0] == 4:
            image = image[:3, :, :]

        with torch.no_grad():
            brain_plane_tensor = transform(image)
            prediction, prediction_label = predict_head_mask(model, brain_plane_tensor)

            if prediction_label == 1:
                brain_plane = pad_to_aspect_ration(image)

                angle = find_angle(prediction)
                # angle = stabilizer_rotate.stabilize(angle)
                prediction = TF.resize(
                    prediction, size=brain_plane.shape[1:], interpolation=T.InterpolationMode.NEAREST
                )
                prediction = TF.rotate(prediction, angle=angle, expand=True, interpolation=T.InterpolationMode.NEAREST)
                brain_plane = TF.rotate(
                    brain_plane, angle=angle, expand=True, interpolation=T.InterpolationMode.BILINEAR
                )

                x1, y1, x2, y2 = masks_to_boxes(prediction)[0].int()
                # x1, y1, x2, y2 = stabilizer_crop.stabilize(x1, y1, x2, y2)
                image = crop(brain_plane, x1, y1, x2, y2, pad=5)

                output_path = output_dir / image_path.name
                image = image.permute(1, 2, 0).numpy()
                image = PIL.Image.fromarray(image)
                image.save(output_path)
                saved_images += 1
            else:
                stabilizer_rotate.reset()
                stabilizer_crop.reset()
    # break


print(f"Number of saved images {saved_images}")